# Track 2: Cardiology Medical Expert Fine-Tuning
This notebook demonstrates domain-specific fine-tuning for Cardiology clinical QA using **Unsloth** and **QLoRA** on the `lmassaron/medical-cardiology-qa` dataset.

### Why Unsloth?
- Up to 5x faster training speeds.
- Up to 60% memory savings, allowing larger models (or larger batch sizes) on a single 16GB VRAM GPU.
- Keeps native model quality without performance degradation.

In [1]:
# Install extra dependencies if needed
# %pip install -U unsloth trl peft bitsandbytes datasets


In [2]:
import os
import torch
import numpy as np
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

max_seq_length = 1024
dtype = None  # None for auto detection (Float16/Bfloat16 based on hardware)
load_in_4bit = True  # NF4 quantization for memory savings

# Check device capabilities
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


## 1. Load Model & Setup LoRA
We load the Unsloth-optimized **Phi-4 Mini Instruct** model (3.8B parameters) and inject PEFT adapters.

In [3]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-mini-instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Setup LoRA adapters targeting all attention and MLP projections
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,  # Unsloth optimized to 0
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


==((====))==  Unsloth 2026.7.5: Fast Phi3 patching. Transformers: 5.5.0. vLLM: 0.25.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


## 2. Load and Prepare the Cardiology QA Dataset
We load the cardiology QA dataset, split it into training/validation, and map the chat messages using the model's chat template.

In [4]:
DATASET_ID = "lmassaron/medical-cardiology-qa"
print(f"Loading {DATASET_ID}...")
dataset = load_dataset(DATASET_ID, split="train")

# Sample evaluation split (10%) reproducible seed
n = len(dataset)
rng = np.random.default_rng(42)
all_idx = rng.permutation(n)
cut = int(n * 0.9)
train_idx, eval_idx = all_idx[:cut], all_idx[cut:]

train_ds = dataset.select(train_idx)
eval_ds = dataset.select(eval_idx)
print(f"Train samples: {len(train_ds)} | Eval samples: {len(eval_ds)}")

# Let's print a sample structure
print("Sample message sequence:", train_ds[0]["messages"])

Loading lmassaron/medical-cardiology-qa...


Train samples: 5149 | Eval samples: 573
Sample message sequence: [{'content': 'You are a knowledgeable medical assistant specializing in cardiology. Answer clinical questions accurately, focusing on diagnostic criteria, treatment guidelines, and pathophysiology.', 'role': 'system'}, {'content': 'What is the mechanism behind the twisting pattern seen in ECG recordings of torsades de pointes?', 'role': 'user'}, {'content': 'The twisting pattern observed in ECG recordings of torsades de pointes is due to the meandering spiral wave formed by the re-entrant circuit, which bends around areas of block in the heart muscle, leading to the characteristic twisting appearance.', 'role': 'assistant'}]


## 3. Formatting Prompts
We map the dataset using the conversation structure.

In [5]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}


train_mapped = train_ds.map(formatting_prompts_func, batched=True)
eval_mapped = eval_ds.map(formatting_prompts_func, batched=True)
print("Prompt Preview:\n", train_mapped[0]["text"])

Prompt Preview:
 <|system|>You are a knowledgeable medical assistant specializing in cardiology. Answer clinical questions accurately, focusing on diagnostic criteria, treatment guidelines, and pathophysiology.<|end|><|user|>What is the mechanism behind the twisting pattern seen in ECG recordings of torsades de pointes?<|end|><|assistant|>The twisting pattern observed in ECG recordings of torsades de pointes is due to the meandering spiral wave formed by the re-entrant circuit, which bends around areas of block in the heart muscle, leading to the characteristic twisting appearance.<|end|>


## 4. Run SFT Trainer using Unsloth
We configure the SFTTrainer using Unsloth's optimized training engine.

In [6]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=0,
        max_steps=100,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] < 8,
        bf16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        output_dir="unsloth-medical-model",
        report_to="none",
        dataset_kwargs={
            "add_special_tokens": False,
        },
    ),
)

# Disable KV cache during training to save memory
model.config.use_cache = False

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/5149 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/573 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,149 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,912,896 of 3,844,934,656 (0.23% trained)


Step,Training Loss,Validation Loss
50,1.000718,0.847546
100,0.846043,0.838491


Unsloth: Restored added_tokens_decoder metadata in unsloth-medical-model/checkpoint-50/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in unsloth-medical-model/checkpoint-100/tokenizer_config.json.


## 5. Save the Adapter
We save the fine-tuned adapter weights to disk.

In [7]:
model.save_pretrained("unsloth-medical-adapter")
tokenizer.save_pretrained("unsloth-medical-adapter")
print("Adapter successfully saved!")

Unsloth: Restored added_tokens_decoder metadata in unsloth-medical-adapter/tokenizer_config.json.


Adapter successfully saved!


## 6. Evaluation and Inference
We put the model into Unsloth's optimized inference mode and run evaluations on our held-out test split. We calculate the average **Perplexity** on the cardiology reference answers (demonstrating how 'surprised' the model is by the clinical ground truth) and print a few example outputs for comparison.

In [8]:
import pandas as pd
from tqdm import tqdm

FastLanguageModel.for_inference(model)  # 2x faster inference

eval_results = []
# We evaluate on a sample of 20 evaluation conversations to run quickly
num_eval_samples = min(20, len(eval_ds))
print(
    f"Calculating perplexity and generating answers for {num_eval_samples} evaluation examples..."
)

for i in tqdm(range(num_eval_samples)):
    messages = eval_ds[i]["messages"]
    user_question = next(m["content"] for m in messages if m["role"] == "user")
    expected_answer = next(m["content"] for m in messages if m["role"] == "assistant")

    # Format inputs for model generation
    encoded = tokenizer.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    # Generate answer
    with torch.no_grad():
        outputs = model.generate(
            input_ids=encoded,
            max_new_tokens=256,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        outputs[0][encoded.shape[1] :], skip_special_tokens=True
    ).strip()

    # Calculate perplexity over target answer tokens
    ref_ids = tokenizer(
        expected_answer, return_tensors="pt", add_special_tokens=False
    ).input_ids.to("cuda")
    full_ids = torch.cat([encoded, ref_ids], dim=1)
    labels = full_ids.clone()
    labels[:, : encoded.shape[1]] = -100  # Mask out prompt tokens

    with torch.no_grad():
        outputs_loss = model(full_ids, labels=labels)
        loss = outputs_loss.loss
        perplexity = torch.exp(loss).item()

    eval_results.append(
        {
            "question": user_question,
            "expected": expected_answer,
            "generated": generated,
            "perplexity": perplexity,
        }
    )

eval_df = pd.DataFrame(eval_results)
print(f"\nAverage Evaluation Perplexity: {eval_df['perplexity'].mean():.4f}")

# Print a sample evaluation comparison
print("\n--- Sample Comparison ---")
print("Question:", eval_df.iloc[0]["question"])
print("\nExpected Answer:", eval_df.iloc[0]["expected"])
print("\nGenerated Answer:", eval_df.iloc[0]["generated"])
print(f"Perplexity: {eval_df.iloc[0]['perplexity']:.4f}")

Calculating perplexity and generating answers for 20 evaluation examples...


  0%|                                                       | 0/20 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  5%|██▎                                            | 1/20 [00:03<01:07,  3.54s/it]

...

 95%|███████████████████████████████████████████▋  | 19/20 [00:20<00:00,  1.07it/s]

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


100%|██████████████████████████████████████████████| 20/20 [00:21<00:00,  1.04it/s]

100%|██████████████████████████████████████████████| 20/20 [00:21<00:00,  1.07s/it]


Average Evaluation Perplexity: 4.4135

--- Sample Comparison ---
Question: What are some connective tissue disorders that increase the risk of aortic dissection?

Expected Answer: Connective tissue disorders that increase the risk of aortic dissection include Marfan syndrome, Ehlers-Danlos syndrome, and Loeys-Dietz syndrome. These conditions affect the structural integrity of the arterial walls.

Generated Answer: Connective tissue disorders such as Marfan syndrome, Ehlers-Danlos syndrome, and Loeys-Dietz syndrome increase the risk of aortic dissection due to their impact on the structural integrity of the aorta.
Perplexity: 1.5121
